# FlyRank Capstone — Refresh-Priority Model

**Question:** among a client's existing content, which pages should an editor prioritize for a
refresh review right now?

**Environment note:** this notebook runs entirely offline against the repo's local starter
dataset (`data/raw/content_refresh_anonymized.csv`, 30,000 rows, 32 pseudonymous clients). ML-07
through ML-09 used the Hugging Face-hosted warehouse table; that host isn't reachable from this
sandbox, so this capstone is scoped to the local dataset. See `work/notebooks/w06_validation_audit.ipynb`
for the leakage audit and the random-vs-grouped-split comparison this notebook relies on.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, f1_score,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df.shape

(30000, 44)

## 1. Target: `is_declining_label`

The FlyRank data dictionary defines `is_declining_label` as `trend_direction == "down"` and
marks it as the sanctioned ML target for this dataset. We use it as-is rather than inventing a
new target, and instead spend the leakage-audit effort on the *feature list*.

In [2]:
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
base_rate = df["is_declining_label"].mean()
print(f"Rows: {len(df)}   Positive rate (declining): {base_rate:.4f}")

Rows: 30000   Positive rate (declining): 0.5421


## 2. Leakage audit and feature set

Excluded: columns that directly define the label, columns whose window overlaps the label's
last30/prev30 comparison, bucketed restatements of already-included features, pseudonymous
identifiers, and provider/model metadata. Full audit table and the disclosed partial-overlap
risk are in `w06_validation_audit.ipynb`.

In [3]:
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
for col in ["search_volume", "competition", "cpc", "word_count", "char_count"]:
    df[col] = df[col].fillna(0)

df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "has_keyword_data", "has_word_count",
]
CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier", "char_count_tier",
]

num_frame = df[NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
cat_frame = pd.get_dummies(df[CATEGORICAL_FEATURES].fillna("unknown").astype(str),
                            prefix=CATEGORICAL_FEATURES, dtype=float)
X = pd.concat([num_frame.reset_index(drop=True), cat_frame.reset_index(drop=True)], axis=1)
y = df["is_declining_label"].reset_index(drop=True)
groups = df["client_id"].reset_index(drop=True)
feature_names = list(X.columns)
print(f"{len(feature_names)} features after encoding")

50 features after encoding


## 3. Baseline — transparent rule

No fitted weights: flag a page if it's **stale** (no update in 90+ days), **was visible**
(90-day impressions >= 300, so it's not simply new/unindexed), and sits at a **weak position**
(average position worse than 10, or no measured position). This mirrors how an editor would
manually triage a list today, so it's a fair thing for a model to have to beat.

In [4]:
stale = (df["days_since_last_update"] >= 90).astype(int)
was_visible = (df["impressions_90d"] >= 300).astype(int)
weak_position = ((df["avg_position"] > 10) | (df["avg_position"] == 0)).astype(int)

df["baseline_score"] = stale * was_visible * (df["impressions_90d"] + 1)
df["baseline_flag"] = (stale & was_visible & weak_position).astype(int)
df["baseline_reason_code"] = np.where(df["baseline_flag"] == 1, "STALE_VISIBLE_WEAK_POSITION", "NO_ACTION")
df["baseline_action"] = np.where(df["baseline_flag"] == 1,
                                  "Refresh content and review for a position slip", "Monitor only")

def precision_at_k(labels, scores, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

print(f"Flagged rows: {int(df['baseline_flag'].sum())}")
print(f"Baseline accuracy (all rows): {accuracy_score(y, df['baseline_flag']):.4f}")
print(f"Baseline precision@50 (all rows): {precision_at_k(y, df['baseline_score'], 50):.4f}")
print(f"Base rate: {base_rate:.4f}")

Flagged rows: 4294
Baseline accuracy (all rows): 0.4893
Baseline precision@50 (all rows): 0.4400
Base rate: 0.5421


The baseline's precision@50 (0.44 on all rows) is actually *below* the base rate (0.54) — a
simple "stale + visible + weak position" rule doesn't beat just guessing "declining" for
everything. That's a real, unflattering baseline result, and it's the fair thing to report: it
means the bar for the model to clear is low, not that the baseline was rigged to lose.

## 4. Validation split — client-grouped

No client appears in both train and test. See `w06_validation_audit.ipynb` for the full
random-vs-grouped comparison; this notebook uses the grouped split throughout.

In [5]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
Xtr, Xte = X.iloc[train_idx], X.iloc[test_idx]
ytr, yte = y.iloc[train_idx], y.iloc[test_idx]
train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

print(f"train_rows={len(Xtr)}  test_rows={len(Xte)}")
print(f"train_clients={len(train_clients)}  test_clients={len(test_clients)}  "
      f"client_overlap={len(train_clients & test_clients)}")

train_rows=23837  test_rows=6163
train_clients=25  test_clients=7  client_overlap=0


## 5. Models — Logistic Regression, Decision Tree, Random Forest

Three models chosen for the question, not to chase a top score: Logistic Regression for a
readable, coefficient-level baseline; Decision Tree for a rule an editor could sanity-check by
eye; Random Forest as an upper bound on what these features can support. `class_weight="balanced"`
because the classes are only mildly imbalanced (54/46) but it costs nothing and avoids a lazy
majority-class fit.

In [6]:
DT_KWARGS = dict(class_weight="balanced", max_depth=5, min_samples_leaf=50, random_state=RANDOM_STATE)

models = {
    "logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "decision_tree": DecisionTreeClassifier(**DT_KWARGS),
    "random_forest": RandomForestClassifier(
        class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
        n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE
    ),
}

dummy = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
dummy.fit(Xtr, ytr)
dummy_acc = accuracy_score(yte, dummy.predict(Xte))

results = {}
fitted = {}
for name, model in models.items():
    model.fit(Xtr, ytr)
    fitted[name] = model
    proba = model.predict_proba(Xte)[:, 1]
    pred = model.predict(Xte)
    results[name] = {
        "roc_auc": roc_auc_score(yte, proba),
        "average_precision": average_precision_score(yte, proba),
        "precision_at_50": precision_at_k(yte.to_numpy(), proba, 50),
        "precision_at_200": precision_at_k(yte.to_numpy(), proba, 200),
        "recall": recall_score(yte, pred),
        "precision": precision_score(yte, pred),
        "f1": f1_score(yte, pred),
        "accuracy": accuracy_score(yte, pred),
    }

baseline_flag_test = df.loc[Xte.index, "baseline_flag"]
baseline_score_test = df.loc[Xte.index, "baseline_score"]
results["baseline_rule"] = {
    "roc_auc": np.nan, "average_precision": np.nan,
    "precision_at_50": precision_at_k(yte.to_numpy(), baseline_score_test.to_numpy(), 50),
    "precision_at_200": precision_at_k(yte.to_numpy(), baseline_score_test.to_numpy(), 200),
    "recall": recall_score(yte, baseline_flag_test),
    "precision": precision_score(yte, baseline_flag_test, zero_division=0),
    "f1": f1_score(yte, baseline_flag_test, zero_division=0),
    "accuracy": accuracy_score(yte, baseline_flag_test),
}
results["dummy_majority_class"] = {"accuracy": dummy_acc}

results_df = pd.DataFrame(results).T
results_df

,roc_auc,average_precision,precision_at_50,precision_at_200,recall,precision,f1,accuracy
logistic_regression,0.611083,0.603705,0.70,0.705,0.628136,0.589919,0.608428,0.586890
decision_tree,0.612273,0.585004,0.50,0.550,0.542394,0.606104,0.572482,0.586078
random_forest,0.599452,0.581615,0.52,0.470,0.580502,0.587215,0.583839,0.577154
baseline_rule,NaN,NaN,0.30,0.380,0.047317,0.385013,0.084276,0.474607
dummy_majority_class,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.510952


In [7]:
best_model_name = max(models, key=lambda n: results[n]["precision_at_50"])
best_model = fitted[best_model_name]
print("Selected model (by precision@50 on held-out clients):", best_model_name)

Selected model (by precision@50 on held-out clients): logistic_regression


## 6. Feature importance + permutation importance

Model-native importance can be misleading for one-hot-encoded categoricals and correlated
numeric features, so we cross-check with permutation importance (shuffle one column at a time on
the held-out clients, measure the drop in ROC AUC).

In [8]:
if best_model_name == "logistic_regression":
    coefs = best_model.named_steps["model"].coef_[0]
    importance_df = pd.DataFrame({"feature": feature_names, "importance": np.abs(coefs)})
else:
    importance_df = pd.DataFrame({"feature": feature_names, "importance": best_model.feature_importances_})
importance_df = importance_df.sort_values("importance", ascending=False).reset_index(drop=True)

perm = permutation_importance(best_model, Xte, yte, n_repeats=10, random_state=RANDOM_STATE,
                               scoring="roc_auc", n_jobs=-1)
perm_df = pd.DataFrame({
    "feature": feature_names,
    "perm_importance_mean": perm.importances_mean,
    "perm_importance_std": perm.importances_std,
}).sort_values("perm_importance_mean", ascending=False).reset_index(drop=True)

print("Top 8 by model importance:")
print(importance_df.head(8).to_string(index=False))
print()
print("Top 8 by permutation importance:")
print(perm_df.head(8).to_string(index=False))

Top 8 by model importance:
               feature  importance
   log_impressions_90d    1.259827
        log_clicks_90d    0.639062
            word_count    0.524189
      content_age_days    0.384430
      log_sessions_90d    0.380498
            char_count    0.335625
          avg_position    0.268688
days_since_last_update    0.236761

Top 8 by permutation importance:
            feature  perm_importance_mean  perm_importance_std
log_impressions_90d              0.074294             0.005782
     log_clicks_90d              0.064623             0.003124
   content_age_days              0.041262             0.002858
   log_sessions_90d              0.029732             0.001536
       avg_position              0.015843             0.002438
         word_count              0.012977             0.002469
        scroll_rate              0.006935             0.002341
         char_count              0.004910             0.001509


Both rankings agree on the top two: `log_impressions_90d` and `log_clicks_90d` dominate. That's
expected and worth naming directly rather than glossing over — these two features have partial
window overlap with the label (see the leakage audit's disclosed risk). The model is leaning
hardest on the feature pair closest to the label's own construction. `content_age_days` and
`log_sessions_90d` are the next tier, which is a cleaner, less-leakage-adjacent signal: older,
lower-session pages tend to be flagged as declining.

## 7. Error analysis

In [9]:
test_frame = df.loc[Xte.index].copy()
test_frame["y_true"] = yte.values
test_frame["y_pred"] = best_model.predict(Xte)
test_frame["y_proba"] = best_model.predict_proba(Xte)[:, 1]

fp = test_frame[(test_frame["y_true"] == 0) & (test_frame["y_pred"] == 1)]
fn = test_frame[(test_frame["y_true"] == 1) & (test_frame["y_pred"] == 0)]

cols = ["content_id", "client_id", "impressions_90d", "avg_position", "days_since_last_update", "y_proba"]
print(f"False positives: {len(fp)}  False negatives: {len(fn)}")
print()
print("Highest-confidence false positives (model said declining, actually wasn't):")
print(fp.sort_values("y_proba", ascending=False).head(3)[cols].to_string(index=False))
print()
print("Lowest-confidence false negatives (model missed, confidently wrong):")
print(fn.sort_values("y_proba", ascending=True).head(3)[cols].to_string(index=False))

False positives: 1375  False negatives: 1171

Highest-confidence false positives (model said declining, actually wasn't):
          content_id         client_id  impressions_90d  avg_position  days_since_last_update  y_proba
content_df71843dcd17 client_8527a891e2            27334          76.4                     103 0.937950
content_5d5653c4eb4f client_4e07408562            15101           5.7                       7 0.937421
content_41baf0722ad9 client_8527a891e2             3115          12.8                     104 0.936325

Lowest-confidence false negatives (model missed, confidently wrong):
          content_id         client_id  impressions_90d  avg_position  days_since_last_update  y_proba
content_d1e915d03c28 client_4e07408562                2          45.0                     104 0.072268
content_c268b1716236 client_e629fa6598                3          41.7                      20 0.076886
content_3d85651f289d client_e629fa6598                6          12.2                  

The false positives above share a pattern: high `impressions_90d` with a very weak
`avg_position` (76.4, 12.8) — the model reads "lots of impressions but poor ranking" as decline
risk, but these pages hadn't actually crossed into the "down" trend bucket yet. The false
negatives are low-impression pages the model doesn't have much signal to work with either way —
consistent with a model that's mostly reasoning from visibility volume, not catching quieter
early-stage declines.

## 8. Ranked action queue (held-out clients only)

In [10]:
queue_cols = ["content_id", "client_id", "impressions_90d", "clicks_90d", "avg_position",
              "days_since_last_update", "content_type", "y_proba", "y_true"]
ranked_queue = test_frame.sort_values("y_proba", ascending=False)[queue_cols].head(200).copy()
ranked_queue["reason_code"] = "MODEL_DECLINE_RISK"
ranked_queue["action"] = "Prioritize for refresh review"

out_dir = Path("../outputs")
out_dir.mkdir(parents=True, exist_ok=True)
ranked_queue.to_csv(out_dir / "capstone_ranked_queue_top200.csv", index=False)
importance_df.to_csv(out_dir / "feature_importance.csv", index=False)
perm_df.to_csv(out_dir / "permutation_importance.csv", index=False)

summary = {
    "random_seed": RANDOM_STATE,
    "n_rows": int(len(df)),
    "n_clients": int(df["client_id"].nunique()),
    "base_rate_is_declining_label": float(base_rate),
    "baseline_precision_at_50_all_rows": precision_at_k(y, df["baseline_score"], 50),
    "train_clients": len(train_clients),
    "test_clients": len(test_clients),
    "client_overlap": len(train_clients & test_clients),
    "train_rows": int(len(Xtr)),
    "test_rows": int(len(Xte)),
    "best_model": best_model_name,
    "model_comparison": results,
}
(out_dir / "capstone_summary.json").write_text(json.dumps(summary, indent=2, default=str))
ranked_queue.head(10)

,content_id,client_id,impressions_90d,clicks_90d,avg_position,days_since_last_update,content_type,y_proba,y_true,reason_code,action
17362,content_c82bc0c24241,client_f369cb89fc,13676,0,4.3,8,keyword article,0.943012,1,MODEL_DECLINE_RISK,Prioritize for refresh review
2646,content_87c007fb5c26,client_f369cb89fc,2463,0,6.6,20,keyword article,0.938856,1,MODEL_DECLINE_RISK,Prioritize for refresh review
10136,content_df71843dcd17,client_8527a891e2,27334,0,76.4,103,keyword article,0.937950,0,MODEL_DECLINE_RISK,Prioritize for refresh review
12869,content_5d5653c4eb4f,client_4e07408562,15101,0,5.7,7,keyword article,0.937421,0,MODEL_DECLINE_RISK,Prioritize for refresh review
20736,content_41baf0722ad9,client_8527a891e2,3115,0,12.8,104,keyword article,0.936325,0,MODEL_DECLINE_RISK,Prioritize for refresh review
9443,content_8ba781dafa55,client_8527a891e2,16156,0,9.0,104,keyword article,0.935670,1,MODEL_DECLINE_RISK,Prioritize for refresh review
4076,content_66458ac1b739,client_8527a891e2,6822,2,2.9,102,keyword article,0.932694,1,MODEL_DECLINE_RISK,Prioritize for refresh review
18587,content_823ea9b9b355,client_f369cb89fc,4369,0,3.9,20,keyword article,0.927610,1,MODEL_DECLINE_RISK,Prioritize for refresh review
27178,content_453722754fea,client_f369cb89fc,140079,16,7.6,20,keyword article,0.926100,1,MODEL_DECLINE_RISK,Prioritize for refresh review
6228,content_e988c1699454,client_8527a891e2,2197,0,21.5,104,keyword article,0.925092,1,MODEL_DECLINE_RISK,Prioritize for refresh review


## Summary

- Target: `is_declining_label`, the data dictionary's sanctioned label — not a re-derived rule.
- Baseline (transparent rule): precision@50 below the base rate — a real, low bar.
- Best model (by precision@50 on held-out clients): see printed output above.
- Top features (`log_impressions_90d`, `log_clicks_90d`) carry a disclosed partial leakage risk;
  treated as a limitation, not hidden.
- Validation: client-grouped, 0 client overlap. Numbers here are what `capstone_report.md` uses.
- Scope: local 30k-row starter dataset only, due to no warehouse network access in this
  environment. Not re-validated at FlyRank's full warehouse scale.